# CPU baseline for the FDTD solver

Runs the same architecture as the PL on the Cortex-A9: 2D TE mode, Q3.13 16 bit fixed point,
round-to-nearest renormalisation, two-pass update (Ey+Ex then Bz), 6 cell cubic PML, cb = -717.

The kernel is written in C and compiled on the board, then run in three configurations:

1. 1 core, scalar (no NEON, no vectorisation)
2. 2 cores, scalar
3. 2 cores with NEON auto-vectorisation

Run all cells top to bottom on the PYNQ.

In [ ]:
c_src = r'''
#include <stdint.h>
#include <string.h>
#include <math.h>

#define GRID 128
#define PML 6
#define CB (-717)

static int16_t ey[GRID][GRID], ex[GRID][GRID], bz[GRID][GRID];
static int32_t ca_ey[GRID][GRID], ca_ex[GRID][GRID], ca_bz[GRID][GRID];
static int iter_count = 0;

static inline int16_t q313(int32_t p) { return (int16_t)((p + 4096) >> 13); }

static int depth(int i) {
    int lo = PML - 1 - i;
    int hi = i - (GRID - PML);
    int d = (lo > 0 ? lo : 0) + (hi > 0 ? hi : 0);
    return d > PML - 1 ? PML - 1 : d;
}

void init(const int32_t *ramp) {
    for (int r = 0; r < GRID; r++)
        for (int c = 0; c < GRID; c++) {
            int dr = depth(r), dc = depth(c);
            ca_ey[r][c] = ramp[dr];
            ca_ex[r][c] = ramp[dc];
            ca_bz[r][c] = ramp[dr > dc ? dr : dc];
        }
}

void reset(void) {
    memset(ey, 0, sizeof ey);
    memset(ex, 0, sizeof ex);
    memset(bz, 0, sizeof bz);
    iter_count = 0;
}

void run(int iters, int nthreads) {
    for (int n = 0; n < iters; n++) {
        #pragma omp parallel for num_threads(nthreads) schedule(static)
        for (int r = 0; r < GRID; r++) {
            for (int c = 0; c < GRID; c++) {
                int32_t b  = bz[r][c];
                int32_t bu = r ? bz[r-1][c] : 0;
                int32_t bl = c ? bz[r][c-1] : 0;
                int16_t eyn = q313(ca_ey[r][c] * ey[r][c] + CB * (b - bu));
                int16_t exn = q313(ca_ex[r][c] * ex[r][c] - CB * (b - bl));
                ey[r][c] = (r == 0 || r == GRID-1) ? 0 : eyn;
                ex[r][c] = (c == 0 || c == GRID-1) ? 0 : exn;
            }
        }

        int32_t s = (int32_t)ey[GRID/2][GRID/2]
                  + (int32_t)(2048.0 * sin(iter_count * 450.0 / 8192.0));
        ey[GRID/2][GRID/2] = (int16_t)s;
        iter_count++;

        #pragma omp parallel for num_threads(nthreads) schedule(static)
        for (int r = 0; r < GRID; r++) {
            for (int c = 0; c < GRID; c++) {
                int32_t eyd = (r < GRID-1) ? ey[r+1][c] : ey[r][c];
                int32_t exr = (c < GRID-1) ? ex[r][c+1] : 0;
                bz[r][c] = q313(ca_bz[r][c] * bz[r][c]
                          + CB * ((eyd - ey[r][c]) - (exr - ex[r][c])));
            }
        }
    }
}
'''

with open('fdtd_bench.c', 'w') as f:
    f.write(c_src)
print('c source written')

In [ ]:
import subprocess, platform

arm = platform.machine() in ('armv7l', 'armv6l')

if arm:
    common       = ['-fopenmp']
    scalar_flags = ['-O2', '-mcpu=cortex-a9', '-mfpu=vfpv3', '-fno-tree-vectorize']
    neon_flags   = ['-O3', '-mcpu=cortex-a9', '-mfpu=neon', '-ftree-vectorize']
else:
    common       = []
    scalar_flags = ['-O2', '-fno-vectorize', '-fno-slp-vectorize']
    neon_flags   = ['-O3']

for name, flags in [('fdtd_scalar.so', scalar_flags), ('fdtd_neon.so', neon_flags)]:
    cmd = ['cc'] + flags + common + ['-shared', '-fPIC', '-o', name, 'fdtd_bench.c', '-lm']
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(name, 'ok' if r.returncode == 0 else r.stderr)

if not arm:
    print('note: not on the pynq, threading and neon flags inactive, numbers are for validation only')

In [ ]:
import ctypes, time

GRID, ITERS = 128, 300
ramp = (ctypes.c_int32 * 6)(8192, 8174, 8045, 7695, 7014, 5892)

def load(path):
    lib = ctypes.CDLL('./' + path)
    lib.init(ramp)
    return lib

def bench(lib, threads):
    lib.reset()
    lib.run(20, threads)
    lib.reset()
    t0 = time.perf_counter()
    lib.run(ITERS, threads)
    dt = time.perf_counter() - t0
    return 3 * GRID * GRID * ITERS / dt

scalar = load('fdtd_scalar.so')
neon   = load('fdtd_neon.so')

results = {
    '1 core scalar':  bench(scalar, 1),
    '2 core scalar':  bench(scalar, 2),
    '2 core + NEON':  bench(neon, 2),
}

FPGA = 2.4e9
print(f"{'config':<16}{'field updates':>16}{'vs fpga':>10}")
for k, v in results.items():
    print(f"{k:<16}{v/1e6:>13.1f} M/s{FPGA/v:>9.1f}x")
print(f"{'fpga 16 lane':<16}{FPGA/1e6:>13.1f} M/s{1.0:>9.1f}x")